# Deployment 5 — RFI flagging working notebook (meeting 2026-08-12)

Load → waterfalls → DPSS flagging → flagged-data checks. Knobs are in CAPS near the top of each cell.

**Data notes (phase C, Jul 15–18):**
- `data/deployment5_filtered/` (local, ~10 GB): science keys only. Phase C files carry `0` (box-gnd), `4` (box-air, 88–92.5 m), `04` (cross). Jul 12–15 files (phases A/B) carry *different* keys — select Jul 15+ patterns when loading `0/4/04`.
- Box-air (`4`) outages: Jul 15 ~18:55–21:30, Jul 16 ~18:42–23:27, Jul 17 ~23:35→. Motor scan: Jul 17 20:28–21:28 UTC.
- Filenames = file-*close* time (usually ~5 s late, up to ~16 min under writer backlog). `header/times` is the precise clock except ~12% bad-sync files (off by days) — fine for plotting, repair before using quantitatively.
- RFSoC comb: **OFF in phase-C data** (spot-checked Jul 16 12:00 & Jul 17 18:00: tone/continuum ≈ 0 dB on key 0, ~+0.5 dB on key 4 — vs +24 dB when on in deployment 4). `FLAG_COMB` in the mask cell turns tone-±1 flagging back on if a comb-on stretch turns up.
- Dropped integrations are written as all-zero rows (whole files can be zero, e.g. Jul 15 12:00); unreadable files leave NaN rows in the loaded arrays.

In [ ]:
%matplotlib widget
import glob
from datetime import datetime, timezone

import h5py
import hera_filters
import numpy as np
import matplotlib.pyplot as plt

## Load

In [ ]:
DATA_DIR = "/home/christian/Documents/research/eigsep/data-analysis/data/deployment5_filtered"

# One glob pattern per chunk to load, e.g.
#   ["corr_20260716*.h5"]                        one day
#   ["corr_20260716*.h5", "corr_20260717*.h5"]   two days
#   ["corr_20260716_0[0-5]*.h5"]                 hours 00-05 of Jul 16
PATTERNS = ["corr_20260717*.h5"]

KEYS = ["0", "4", "04"]  # any subset (phase C files only have these)
TIME_AVG = 8             # integrations averaged at load (1 = full 0.54 s resolution)
NCHAN = 1024

files = np.array(sorted(set(sum([glob.glob(f"{DATA_DIR}/{p}") for p in PATTERNS], []))))
print(f"{len(files)} files")
dates = np.array([f.split("corr_")[1][:8] for f in files])
for d in np.unique(dates):
    print(f"  {d}: {(dates == d).sum()} files")

In [ ]:
def fname_unix(fn):
    """File-close time from the filename. Typically ~5 s after the last
    integration; up to ~16 min late when the writer backlogs."""
    s = fn.split("corr_")[-1][:16]  # YYYYMMDD_HHMMSSZ
    dt = datetime.strptime(s, "%Y%m%d_%H%M%SZ").replace(tzinfo=timezone.utc)
    return dt.timestamp()

def scan(files):
    """Integrations + first acc_cnt per file (0/-1 = unreadable). Metadata-only."""
    n = np.zeros(len(files), dtype=int)
    a0 = np.full(len(files), -1.0)
    for i, fn in enumerate(files):
        try:
            with h5py.File(fn, "r") as f:
                n[i] = f["header/times"].shape[0]
                a0[i] = f["header/acc_cnt"][0]
        except Exception:
            pass
    return n, a0

nrows_in, acc0 = scan(files)
# Backlog flushes write several files in the same second with '-1','-2',...
# name suffixes; plain sorted order puts '-1' (later data) BEFORE the base
# file. Break filename-time ties with acc_cnt to restore true time order.
fname_t = np.array([fname_unix(f) for f in files])
order = np.lexsort((acc0, fname_t))
files, nrows_in, fname_t = files[order], nrows_in[order], fname_t[order]

bad = (nrows_in == 0).sum()
nout_per_file = nrows_in // TIME_AVG
n_rows = int(nout_per_file.sum())

bytes_per_row = sum(8 if len(k) == 2 else 4 for k in KEYS) * NCHAN
est_gb = n_rows * bytes_per_row / 1e9
avail_gb = int(open("/proc/meminfo").read().split("MemAvailable:")[1].split()[0]) / 1e6
print(f"{n_rows} integrations after averaging ({bad} unreadable files)")
print(f"estimated array memory: {est_gb:.1f} GB, available now: {avail_gb:.1f} GB")
if est_gb > 0.5 * avail_gb:
    print("WARNING: >50% of available RAM -- increase TIME_AVG or load fewer days/keys")

In [ ]:
# Row time axis uses filename times -- good enough for labeling/day boundaries.
def load(files, keys, nout_per_file, time_avg):
    n_rows = int(nout_per_file.sum())
    data = {
        k: np.full((n_rows, NCHAN), np.nan,
                   dtype=np.complex64 if len(k) == 2 else np.float32)
        for k in keys
    }
    times = np.full(n_rows, np.nan)
    starts = np.concatenate([[0], np.cumsum(nout_per_file)])
    skipped = 0
    for i, fn in enumerate(files):
        nout = nout_per_file[i]
        if nout == 0:
            continue
        keep, r = nout * time_avg, starts[i]
        times[r : r + nout] = fname_unix(fn)
        try:
            with h5py.File(fn, "r") as f:
                for k in keys:
                    arr = f["data"][k][:keep]
                    if len(k) == 2:  # cross stored as (..., 2) = (re, im) int32
                        arr = (arr[..., 0].astype(np.float32)
                               + 1j * arr[..., 1].astype(np.float32))
                    else:
                        arr = arr.astype(np.float32)
                    data[k][r : r + nout] = arr.reshape(nout, time_avg, NCHAN).mean(1)
        except Exception:
            skipped += 1  # rows of this file stay NaN
        if i % 200 == 0:
            print(f"\r{i}/{len(files)}", end="")
    print(f"\r{len(files)}/{len(files)} done, {skipped} files skipped (rows left NaN)")
    return data, times

with h5py.File(files[nrows_in.argmax()], "r") as f:
    freqs = f["header/freqs"][:]                              # MHz
    df_mhz = f["header"].attrs["dfreq"]                       # 0.244 MHz
    dt = f["header"].attrs["integration_time"] * TIME_AVG     # s per loaded row
data, times = load(files, KEYS, nout_per_file, TIME_AVG)

t0 = datetime.fromtimestamp(times[0], tz=timezone.utc)
t1 = datetime.fromtimestamp(times[-1], tz=timezone.utc)
print(f"{t0:%Y-%m-%d %H:%M} to {t1:%Y-%m-%d %H:%M} UTC, dt = {dt:.2f} s/row")

## Waterfalls

In [ ]:
DISPLAY_MAX_ROWS = 4000  # waterfalls are row-binned to at most this before imshow

def rebin_rows(a, max_rows=None):
    """Mean over row blocks so imshow never holds a huge array."""
    max_rows = max_rows or DISPLAY_MAX_ROWS
    fac = int(np.ceil(len(a) / max_rows))
    if fac <= 1:
        return a
    keep = (len(a) // fac) * fac
    return np.nanmean(a[:keep].reshape(-1, fac, a.shape[1]), axis=1)

def day_lines(ax):
    """Dotted line + date label at each UTC day boundary."""
    edges = np.flatnonzero(np.diff(times // 86400)) + 1
    for row in edges:
        ax.axhline(row, color="w", ls=":", lw=0.8)
    for row in np.concatenate([[0], edges]).astype(int):
        lab = datetime.fromtimestamp(times[row], tz=timezone.utc).strftime("%b %d")
        ax.text(0.01, row, " " + lab, color="w", fontsize=8, va="top",
                transform=ax.get_yaxis_transform(),
                bbox={"facecolor": "k", "alpha": 0.4, "pad": 1, "edgecolor": "none"})
    ax.set_xlabel("Frequency [MHz]")
    ax.set_ylabel(f"Integration (x{TIME_AVG} avg)")

def wfall(a, vmin=None, vmax=None, cmap="plasma", title="", cbar=""):
    fig, ax = plt.subplots(figsize=(9, 5), layout="constrained")
    extent = [freqs.min(), freqs.max(), len(a), 0]
    im = ax.imshow(rebin_rows(a), aspect="auto", cmap=cmap, extent=extent,
                   interpolation="none", vmin=vmin, vmax=vmax)
    fig.colorbar(im, ax=ax, label=cbar)
    ax.set_title(title)
    day_lines(ax)
    plt.show()
    return

def log10_(a):
    with np.errstate(divide="ignore", invalid="ignore"):
        return np.log10(np.abs(a))

In [ ]:
if "0" in data:
    wfall(log10_(data["0"]), 4, 6.6, title="key 0 (box-gnd)", cbar="log10(power)")
if "4" in data:
    wfall(log10_(data["4"]), 4, 6.6, title="key 4 (box-air)", cbar="log10(power)")
if "04" in data:
    wfall(log10_(data["04"]), 3, 6, title="|04|", cbar="log10|cross|")
    wfall(np.angle(data["04"]), -np.pi, np.pi, cmap="twilight",
          title="arg(04)", cbar="phase [rad]")

## DPSS flagging

Same workflow as deployment-4 `dpss_fitting`:
1. base mask (dead rows/pixels; comb tones ±1 only if `FLAG_COMB`) on a band-restricted chunk,
2. DPSS bases in freq (delay half-width) and time (smoothness timescale),
3. 2D fit → smooth model `dmdl` + radiometer noise model `nmdl`,
4. iterate: re-flag where `|data − dmdl| > thresh × nmdl`, refit, lower `thresh`.

Slice the band instead of zero-weighting wide chunks — wide contiguous zero-weight bands condition the fit badly. Start on a time chunk (`TSEL`) before committing to a full day: the 2D fit is the slow step.

In [ ]:
KEY = "0"
BAND = (50, 225)      # MHz kept for fitting (slice, don't zero-weight wide bands)
TSEL = slice(0, 2000)  # rows to work on; slice(None) = everything loaded
FLAG_COMB = False     # comb is OFF in phase-C data (tone/continuum ~0 dB; +24 dB
                      # in deployment 4) -- set True to flag tones +-COMB_SPILL
COMB_SPILL = 1

f_ix = (freqs >= BAND[0]) & (freqs <= BAND[1])
fb = freqs[f_ix]
# NaN rows (skipped files) must be zero-filled: NaN * weight 0 still breaks the fit
d = np.nan_to_num(np.abs(data[KEY][TSEL])[:, f_ix]).astype(np.float64)
tb = times[TSEL]

mask0 = np.ones(d.shape, dtype=bool)
if FLAG_COMB:
    # comb tones sit in every 16th channel of the full 1024-ch grid
    comb = np.zeros(NCHAN, dtype=bool)
    for off in range(-COMB_SPILL, COMB_SPILL + 1):
        comb[np.clip(np.arange(0, NCHAN, 16) + off, 0, NCHAN - 1)] = True
    mask0[:, comb[f_ix]] = False
mask0 &= d > 0                 # dropped integrations are all-zero rows
mask0[d.max(axis=1) <= 0] = False

print(f"{d.shape} rows x chans, base mask flags {1 - mask0.mean():.1%}")

In [ ]:
DLY_HW = 300e-9  # s -- freq-axis half-width (h_ant=100 m -> ~333 ns light travel)
TAU = 100.0      # s -- time-axis smoothness timescale (deployment-4 value)

t_sec = np.arange(d.shape[0]) * dt  # nominal uniform grid; file gaps are approximate
Af = hera_filters.dspec.dpss_operator(fb * 1e6, filter_centers=[0],
                                      filter_half_widths=[DLY_HW],
                                      eigenval_cutoff=[1e-9])[0].real
At = hera_filters.dspec.dpss_operator(t_sec, filter_centers=[0],
                                      filter_half_widths=[1 / TAU],
                                      eigenval_cutoff=[1e-9])[0].real
print(f"At: {At.shape}, Af: {Af.shape}")

In [ ]:
# Quick 1D sanity check on single slices before the 2D fit
def plot_spec(y, m, x=None, model=None, noise_model=None, log=True, xlabel="Frequency [MHz]"):
    x = fb if x is None else x
    plt.figure()
    plt.plot(x, y, label="Data", alpha=0.5)
    if model is not None:
        plt.plot(x, model, label="Model")
        plt.plot(x, np.abs((y - model) * m), label="|Residual| (masked)")
        if noise_model is not None:
            plt.plot(x, noise_model, label="Noise Model")
    else:
        plt.plot(x, np.where(m, y, np.nan), label="Masked")
    if log:
        plt.yscale("log")
    plt.xlabel(xlabel)
    plt.ylabel("Amplitude")
    plt.legend()
    plt.show()

def fit_slice(data, mask, ix=400, axis="freq"):
    A = Af if axis == "freq" else At
    y = data[ix] if axis == "freq" else data[:, ix]
    ym = mask[ix] if axis == "freq" else mask[:, ix]
    F = hera_filters.dspec.fit_solution_matrix(ym.astype("float64"), A)
    return y, ym, A @ (F @ y)

IX = d.shape[0] // 2
y, ym, mdl = fit_slice(d, mask0, ix=IX)
plot_spec(y, ym, model=mdl)

y, ym, mdl = fit_slice(d, mask0, ix=400, axis="time")
plot_spec(y, ym, x=t_sec, model=mdl, xlabel="Time [s]")

In [ ]:
def make_mask(data, thresh, data_model, noise_model):
    """True (keep) where |data - model| < thresh * noise."""
    return np.abs(data - data_model) / noise_model < thresh

def make_mdls(data, mask):
    fit, meta = hera_filters.dspec.sparse_linear_fit_2D(
        data=data, weights=mask.astype(float),
        axis_1_basis=At, axis_2_basis=Af, precondition_solver=False)
    print(meta)
    dmdl = At @ fit @ Af.T
    # radiometer estimate; real noise exceeds this beyond ~10 s (gain wander)
    nmdl = dmdl / np.sqrt(2 * df_mhz * 1e6 * dt)
    return dmdl, nmdl

def plot_mask(mask):
    plt.figure()
    plt.imshow(mask.astype(int), aspect="auto", interpolation="none",
               extent=[fb[0], fb[-1], len(mask), 0])
    plt.xlabel("Frequency [MHz]")
    plt.ylabel("Row")
    plt.title(f"mask ({1 - mask.mean():.1%} flagged)")
    plt.show()

def step(data, mask=None, dmdl_prev=None, nmdl_prev=None, thresh=None):
    if mask is None:
        mask = make_mask(data, thresh, dmdl_prev, nmdl_prev) & mask0
        plot_mask(mask)
    dmdl, nmdl = make_mdls(data, mask)
    plot_spec(data[IX], mask[IX], model=dmdl[IX], noise_model=nmdl[IX])
    return dmdl, nmdl, mask

In [ ]:
# First fit with the base mask...
dmdl, nmdl, mask = step(d, mask=mask0)

In [ ]:
# ...then iterate this cell, lowering thresh each pass (e.g. 100 -> 20 -> 6)
dmdl, nmdl, mask = step(d, dmdl_prev=dmdl, nmdl_prev=nmdl, thresh=100)

In [ ]:
# Flag occupancy: which channels / times are getting cut
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(9, 6), layout="constrained")
ax1.plot(fb, 1 - mask.mean(axis=0))
ax1.set_xlabel("Frequency [MHz]")
ax1.set_ylabel("Flag fraction")
ax2.plot(1 - mask.mean(axis=1))
ax2.set_xlabel("Row")
ax2.set_ylabel("Flag fraction")
plt.show()

## Flagged-data checks

Residual/noise ratio: if the flagged residuals are noise-like, the histogram is a
Gaussian; σ ≈ 1 means radiometer-limited (expect σ > 1 from gain wander /
multipath fringes on long timescales).

In [ ]:
res_ratio = np.where(mask, (d - dmdl) / nmdl, np.nan)

fig, ax = plt.subplots(figsize=(9, 5), layout="constrained")
im = ax.imshow(rebin_rows(res_ratio), aspect="auto", cmap="coolwarm",
               extent=[fb[0], fb[-1], len(res_ratio), 0],
               interpolation="none", vmin=-5, vmax=5)
fig.colorbar(im, ax=ax, label="(data - model) / noise")
ax.set_xlabel("Frequency [MHz]")
ax.set_ylabel("Row")
plt.show()

In [ ]:
def hist_gauss(x, bins=np.arange(-8, 8, 0.1), log=True):
    """Histogram + unit Gaussian + robust-fit Gaussian (median / 1.4826*MAD)."""
    x = x[np.isfinite(x)]
    med = np.median(x)
    sig = 1.4826 * np.median(np.abs(x - med))
    print(f"mean={x.mean():.3f} std={x.std():.3f} | median={med:.3f} MAD-sigma={sig:.3f}")
    gx = np.linspace(bins[0], bins[-1], 1000)
    plt.figure()
    plt.hist(x, bins=bins, density=True, histtype="step", lw=1.5, label="data")
    plt.plot(gx, np.exp(-gx**2 / 2) / np.sqrt(2 * np.pi), "k--", label="N(0, 1)")
    plt.plot(gx, np.exp(-((gx - med) / sig) ** 2 / 2) / (sig * np.sqrt(2 * np.pi)),
             "r-", label=f"N({med:.2f}, {sig:.2f}$^2$)")
    if log:
        plt.yscale("log")
        plt.ylim(1e-6, 1)
    plt.xlabel("(data - model) / noise")
    plt.ylabel("density")
    plt.legend()
    plt.show()
    return med, sig

med, sig = hist_gauss(res_ratio.ravel())

In [ ]:
# Per-channel scatter vs the radiometer expectation
sig_ch = 1.4826 * np.nanmedian(np.abs(res_ratio - np.nanmedian(res_ratio, axis=0)), axis=0)
plt.figure()
plt.plot(fb, sig_ch)
plt.axhline(1, color="k", ls="--", label="radiometer")
plt.xlabel("Frequency [MHz]")
plt.ylabel("MAD-sigma of (data-model)/noise")
plt.legend()
plt.show()

In [ ]:
# Delay-space residuals (bh7 window, masked): leftover RFI shows as ridges
window = hera_filters.dspec.gen_window("bh7", fb.size)
dly_ns = np.fft.fftshift(np.fft.fftfreq(fb.size, d=df_mhz * 1e6)) * 1e9
vis = np.fft.fftshift(np.fft.fft(window[None, :] * (d - dmdl) * mask, axis=1), axes=1)

fig, ax = plt.subplots(figsize=(9, 5), layout="constrained")
im = ax.imshow(rebin_rows(log10_(vis)), aspect="auto", cmap="plasma",
               extent=[dly_ns[0], dly_ns[-1], len(vis), 0], interpolation="none")
fig.colorbar(im, ax=ax, label="log10|FFT residual|")
ax.set_xlabel("Delay [ns]")
ax.set_ylabel("Row")
plt.show()

## Scratch

In [ ]:
plt.close("all")  # run when widget figures pile up -- each holds its arrays